### Stage 4: Outcome label

For each **notification × HS4 × exporter** triple, define $Y = 1$ if, over the **six months** after the notification, US imports of the targeted HS4 product from exporter $e$ fall by at least \theta **log-points** relative to the overall US import change for that HS4 (product-level shock control), and the exporter $e$'s overall import change across all products (exporter-wide shock control). This two-way adjustment removes broad product-level shocks and exporter-wide disruptions, so the label captures **notification-specific** trade disruptions.

Formally: $$Y_{npe} = \mathbf{1} \Bigg\{ \Delta \widetilde{M}_{npe} \leq - \frac{\theta}{100} \Bigg\}$$ 
where
- Using $\theta = 20$ is a baseline threshold for a meaningful trade disruption, not trivial fluctuation; robustness to alternative cutoffs ($10, 15, 25$)
- $Y_{npe} = 1$ means notification $i$ is labeled import-reducing
- $\Delta \widetilde{M}_{npe}$ is the abnormal post-notification import change detrended by both product and exporter shocks, with 6-month time differences: $$\Delta \widetilde{M}_{npe}^{(0,6)} = \Delta \log M_{pe, [t_n, t_n+6]} - \Delta \log M_{p, [t_n, t_n+6]} - \Delta \log M_{e, [t_n, t_n+6]}$$

**4.1. Data loading**

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Project'),
    Path('/mnt/g/My Drive/Project'),
    Path(r'G:/My Drive/Project'),
    Path.cwd(),
]
PROJECT_ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), Path.cwd())
CLEAN = PROJECT_ROOT / 'clean_out'

# ---- Parameters ----
THETA = 0.20                          # 20 log-point cutoff
K     = 6                             # 6-month post-notification window
THR   = np.log(1 - THETA)            # ≈ -0.2231

# ---- Load data ----
panel = pd.read_csv(CLEAN / 'hs4_imports_with_notif_exposure.csv',
                     dtype={'country': str, 'hs4': str})
panel['date'] = pd.to_datetime(panel['date'])

notif_hs4 = pd.read_csv(CLEAN / 'notif_hs4_detailed.csv', dtype={'hs4': str})
notif_hs4['t0'] = pd.to_datetime(
    notif_hs4['year'].astype(str) + '-' +
    notif_hs4['month'].astype(str).str.zfill(2) + '-01',
    errors='coerce')
notif_hs4 = notif_hs4.dropna(subset=['t0', 'notified_document', 'hs4']).copy()

# Renormalize mapping weights within each notification
notif_hs4['w'] = pd.to_numeric(notif_hs4['w'], errors='coerce').fillna(0)
den = notif_hs4.groupby('notified_document')['w'].transform('sum')
notif_hs4 = notif_hs4[den > 0].copy()
notif_hs4['w'] = notif_hs4['w'] / den

print('Using project root:', PROJECT_ROOT)
print(f'Panel rows: {len(panel):,}  |  Notif-HS4 rows: {len(notif_hs4):,}')
print(f'Threshold: {THR:.4f} log-points  |  Window: {K} months')

Using project root: G:\My Drive\Project
Panel rows: 522,855  |  Notif-HS4 rows: 28,384
Threshold: -0.2231 log-points  |  Window: 6 months


**4.2. Two-way adjusted "abnormal import decline" label**

In [3]:
# ================================================================
# For each (notification, HS4, exporter):
#   delta_cp  = log change in exporter-product imports (3-mo pre → 6-mo post)
#   delta_hs4 = log change in total US imports for that HS4 (same windows)
#   delta_exp = log change in exporter's total imports across all products
#   Y = 1  if  (delta_cp - delta_hs4 <= THR)  AND  (delta_cp - delta_exp <= THR)
# ================================================================

notif_map = notif_hs4[['notified_document', 'hs4', 't0', 'w']].copy()
notif_map = notif_map.rename(columns={'t0': 'notif_month'})
wden = notif_map.groupby('notified_document')['w'].transform('sum')
notif_map = notif_map[wden > 0].copy()
notif_map['w'] = notif_map['w'] / wden

# Helper: event-time tau
def add_tau(d):
    d = d.copy()
    d['tau'] = (
        d['date'].dt.to_period('M') - d['notif_month'].dt.to_period('M')
    ).apply(lambda x: x.n)
    return d

# -------- 1) Exporter × product series --------
cp = panel[['country', 'hs4', 'date', 'import_value']].merge(
    notif_map, on='hs4', how='inner')
cp = add_tau(cp)
cp = cp[cp['tau'].between(-3, K)].copy()
cp['w_imp'] = cp['w'] * cp['import_value']

cp_agg = (
    cp.groupby(['notified_document', 'country', 'hs4', 'tau'], as_index=False)
      .agg(imp=('w_imp', 'sum'))
)
cp_base = (cp_agg[cp_agg['tau'].between(-3, -1)]
           .groupby(['notified_document', 'country', 'hs4'], as_index=False)['imp'].mean())
cp_post = (cp_agg[cp_agg['tau'].between(1, K)]
           .groupby(['notified_document', 'country', 'hs4'], as_index=False)['imp'].mean())
cp_wide = cp_base.merge(cp_post,
    on=['notified_document', 'country', 'hs4'], suffixes=('_base', '_post'), how='inner')
cp_wide['delta_cp'] = np.log1p(cp_wide['imp_post']) - np.log1p(cp_wide['imp_base'])

# -------- 2) Product-level control (all exporters, same HS4) --------
hs4_tot = (panel.groupby(['hs4', 'date'], as_index=False)['import_value']
               .sum().rename(columns={'import_value': 'imp_hs4'}))
hs4_tot = hs4_tot.merge(notif_map[['notified_document', 'hs4', 'notif_month', 'w']],
                         on='hs4', how='inner')
hs4_tot = add_tau(hs4_tot)
hs4_tot = hs4_tot[hs4_tot['tau'].between(-3, K)].copy()
hs4_tot['w_imp'] = hs4_tot['w'] * hs4_tot['imp_hs4']

hs4_agg = (hs4_tot.groupby(['notified_document', 'hs4', 'tau'], as_index=False)
                   .agg(imp=('w_imp', 'sum')))
hs4_base = (hs4_agg[hs4_agg['tau'].between(-3, -1)]
            .groupby(['notified_document', 'hs4'], as_index=False)['imp'].mean())
hs4_post = (hs4_agg[hs4_agg['tau'].between(1, K)]
            .groupby(['notified_document', 'hs4'], as_index=False)['imp'].mean())
hs4_wide = hs4_base.merge(hs4_post,
    on=['notified_document', 'hs4'], suffixes=('_base', '_post'), how='inner')
hs4_wide['delta_hs4'] = np.log1p(hs4_wide['imp_post']) - np.log1p(hs4_wide['imp_base'])

# -------- 3) Exporter-level control (same exporter, all products) --------
exp_tot = (panel.groupby(['country', 'date'], as_index=False)['import_value']
               .sum().rename(columns={'import_value': 'imp_exp'}))

# Cross with notifications via cp keys to get proper event-time
exp_keys = cp[['notified_document', 'country', 'hs4', 'notif_month', 'w']].drop_duplicates()
exp_tot = exp_tot.merge(exp_keys, on='country', how='inner')
exp_tot = add_tau(exp_tot)
exp_tot = exp_tot[exp_tot['tau'].between(-3, K)].copy()
exp_tot['w_imp'] = exp_tot['w'] * exp_tot['imp_exp']

exp_agg = (exp_tot.groupby(['notified_document', 'country', 'hs4', 'tau'], as_index=False)
                  .agg(imp=('w_imp', 'sum')))
exp_base = (exp_agg[exp_agg['tau'].between(-3, -1)]
            .groupby(['notified_document', 'country', 'hs4'], as_index=False)['imp'].mean())
exp_post = (exp_agg[exp_agg['tau'].between(1, K)]
            .groupby(['notified_document', 'country', 'hs4'], as_index=False)['imp'].mean())
exp_wide = exp_base.merge(exp_post,
    on=['notified_document', 'country', 'hs4'], suffixes=('_base', '_post'), how='inner')
exp_wide['delta_exp'] = np.log1p(exp_wide['imp_post']) - np.log1p(exp_wide['imp_base'])

# -------- 4) Two-way adjusted label --------
ev = cp_wide.merge(
    hs4_wide[['notified_document', 'hs4', 'delta_hs4']],
    on=['notified_document', 'hs4'], how='left'
).merge(
    exp_wide[['notified_document', 'country', 'hs4', 'delta_exp']],
    on=['notified_document', 'country', 'hs4'], how='left'
)

ev['delta_vs_hs4'] = ev['delta_cp'] - ev['delta_hs4']
ev['delta_vs_exp'] = ev['delta_cp'] - ev['delta_exp']
ev['Y_struct_abn'] = (
    (ev['delta_vs_hs4'] <= THR) & (ev['delta_vs_exp'] <= THR)
).astype(int)

print(f'Two-way abnormal label  (theta={THETA}, window={K} months)')
print(f'  pos_rate = {ev["Y_struct_abn"].mean():.4f}  |  n = {len(ev):,}')
print(ev[['delta_vs_hs4', 'delta_vs_exp']].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

Two-way abnormal label  (theta=0.2, window=6 months)
  pos_rate = 0.1814  |  n = 52,033
      delta_vs_hs4  delta_vs_exp
0.10     -0.738818     -0.890677
0.25     -0.245338     -0.375047
0.50      0.000000      0.021606
0.75      0.271874      0.466396
0.90      0.837610      1.108643


**4.3. Save labels file**

In [7]:
# Keep text metadata from notif_hs4 for downstream modelling
text_cols = notif_hs4[['notified_document', 'title', 'description']].drop_duplicates('notified_document')
labels = ev[['notified_document', 'country', 'hs4', 'Y_struct_abn']].copy()
labels = labels.merge(text_cols, on='notified_document', how='left')

# Save
out_path = CLEAN / 'notif_labels_hs4.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w', encoding='utf-8', newline='') as f:
    labels.to_csv(f, index=False)

print(f'Saved: {out_path}')
print(f'  Rows (notif × HS4 × exporter): {len(labels):,}')
print(f'  Unique notifications: {labels["notified_document"].nunique():,}')
print(f'  Share Y=1: {labels["Y_struct_abn"].mean():.4f}')

Saved: G:\My Drive\Project\clean_out\notif_labels_hs4.csv
  Rows (notif × HS4 × exporter): 52,033
  Unique notifications: 2,408
  Share Y=1: 0.1814
